# EfficientNet-B3 Binar — versiune corectată

**Ce s-a schimbat față de varianta ta și de ce:**

1. **Antrenare în două faze** (`FREEZE_BACKBONE_EPOCHS = 3`). În faza 1 backbone-ul e înghețat și se antrenează doar capul nou, cu LR mare (`1e-3`). În faza 2 se dezgheață tot, cu LR diferențiat: backbone `1e-4`, cap `3e-4`. Asta rezolvă cauza principală a platoului — capul random nu mai strică feature-urile ImageNet, iar `1e-5` uniform de dinainte era prea mic ca să învețe capul.
2. **Augmentare activată** (flip, rotație, color jitter) — era complet comentată.
3. **`num_workers=0`** — propriul tău comentariu avertiza că pe Windows `num_workers>0` îngheață la epoca 2, dar codul folosea `8`. Acum e consecvent.
4. **AUC raportat și monitorizat** — early stopping și checkpoint pe AUC de validare (metrică potrivită pentru context medical), nu doar accuracy.
5. **Prag de decizie ales pe validare** (Youden's J) și aplicat consecvent pe test — înainte aveai `0.5` la validare și `0.3` la test, ceea ce nu e reproductibil.
6. **Etichete forțate la `float`** în buclă — defensiv, ca `BCEWithLogitsLoss` să nu crape indiferent de ce returnează `binary_dataset.py`.
7. **Celulă opțională de sanity-check** (overfit pe 100 de imagini) ca să verifici că pipeline-ul învață înainte să aștepți ore.

> **Notă despre Ben Graham:** datele tale sunt deja în `processed_by_me/...`, deci preprocesarea de fundus e lăsată **dezactivată** (`USE_BEN_GRAHAM = False`). Dacă platoul persistă după aceste schimbări, pune-o pe `True` și verifică mai întâi pe câteva imagini cum arată rezultatul (nu o aplica orbește peste date deja procesate).

In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Importam DOAR clasa de Dataset din fisierul nostru extern
from binary_dataset import BinaryRetinopathyDataset

print("=== ÎNCĂRCARE DATE ȘI DATALOADERS ===")

DATA_DIR = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_binar'
IMG_SIZE = 300          # rezolutia nativa a EfficientNet-B3
BATCH_SIZE = 32         # scade la 16/8 daca iei CUDA Out of Memory
USE_BEN_GRAHAM = False  # vezi nota din primul cell inainte sa activezi


# --- Preprocesare optionala specifica de fundus (Ben Graham, EyePACS 2015) ---
class BenGrahamPreprocess:
    """Crop circular + scaderea mediei locale de culoare (Gaussian blur).
    Foloseste-o DOAR daca imaginile nu sunt deja preprocesate astfel."""
    def __init__(self, sigma_scale=10, crop_tol=7):
        self.sigma_scale = sigma_scale
        self.crop_tol = crop_tol

    def _crop_to_circle(self, img):
        import cv2
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray > self.crop_tol
        if mask.sum() == 0:
            return img
        coords = np.argwhere(mask)
        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return img[y0:y1, x0:x1]

    def __call__(self, img):
        import cv2
        from PIL import Image
        arr = np.array(img.convert("RGB"))
        arr = self._crop_to_circle(arr)
        s = max(arr.shape[0], arr.shape[1]) / self.sigma_scale
        blurred = cv2.GaussianBlur(arr, (0, 0), s)
        arr = cv2.addWeighted(arr, 4, blurred, -4, 128)
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


_pre = [BenGrahamPreprocess()] if USE_BEN_GRAHAM else []
_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# 1. Transformari - AUGMENTAREA E ACUM ACTIVA pe train
train_transform = transforms.Compose(_pre + [
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    _norm,
])

test_transform = transforms.Compose(_pre + [
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    _norm,
])

# 2. Dataseturi
train_ds = BinaryRetinopathyDataset(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_ds   = BinaryRetinopathyDataset(os.path.join(DATA_DIR, 'val'),   transform=test_transform)
test_ds  = BinaryRetinopathyDataset(os.path.join(DATA_DIR, 'test'),  transform=test_transform)

# 3. DataLoadere - num_workers=0 pe Windows (vezi avertismentul tau despre inghetare)
NUM_WORKERS = 7
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("DataLoaderele au fost create cu succes!")
print(f"Numar de batch-uri pe antrenament: {len(train_loader)}")
print(f"Ben Graham: {'ACTIV' if USE_BEN_GRAHAM else 'dezactivat'}")


=== ÎNCĂRCARE DATE ȘI DATALOADERS ===
DataLoaderele au fost create cu succes!
Numar de batch-uri pe antrenament: 3021
Ben Graham: dezactivat


## Model + configurație în două faze

Construim o singură dată modelul. Optimizatorul îl construim pe faze: în faza 1 doar capul, în faza 2 două grupuri de parametri cu LR diferit.

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Antrenam pe: {device}")

# --- Hiperparametri ---
HEAD_LR_PHASE1     = 1e-3   # LR mare pentru capul random, cu backbone inghetat
BACKBONE_LR_PHASE2 = 1e-4   # LR mic pentru backbone-ul pre-antrenat
HEAD_LR_PHASE2     = 3e-4   # capul ramane putin mai "rapid" decat backbone-ul
WEIGHT_DECAY       = 1e-4
DROPOUT            = 0.40
FREEZE_BACKBONE_EPOCHS  = 3   # cate epoci tinem backbone-ul inghetat (faza 1)
EARLY_STOPPING_PATIENCE = 5
USE_POS_WEIGHT     = False     # setul binarizat e deja ~50/50

# --- EfficientNet-B3 pre-antrenat ---
model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=DROPOUT),
    nn.Linear(num_features, 1)
)
model = model.to(device)

# Verificam distributia claselor
num_sanatosi = len(os.listdir(os.path.join(DATA_DIR, 'train', 'sanatos')))
num_bolnavi  = len(os.listdir(os.path.join(DATA_DIR, 'train', 'bolnav')))
print(f"Clase: sanatos={num_sanatosi}, bolnav={num_bolnavi}")

if USE_POS_WEIGHT:
    pos_weight = torch.tensor([num_sanatosi / max(num_bolnavi, 1)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    print(f"pos_weight activ: {pos_weight.item():.4f}")
else:
    criterion = nn.BCEWithLogitsLoss()
    print("pos_weight dezactivat (set echilibrat).")


# --- Helpers pentru inghet/dezghet si construirea optimizatorului pe faze ---
def set_backbone_trainable(m, trainable):
    for name, p in m.named_parameters():
        if not name.startswith("classifier"):
            p.requires_grad = trainable

def split_params(m):
    head = [p for n, p in m.named_parameters() if n.startswith("classifier")]
    backbone = [p for n, p in m.named_parameters() if not n.startswith("classifier")]
    return head, backbone

def build_optimizer_phase1(m):
    set_backbone_trainable(m, False)  # backbone inghetat
    head, _ = split_params(m)
    opt = optim.AdamW(head, lr=HEAD_LR_PHASE1, weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
    return opt, sch

def build_optimizer_phase2(m):
    set_backbone_trainable(m, True)   # tot dezghetat
    head, backbone = split_params(m)
    opt = optim.AdamW([
        {"params": backbone, "lr": BACKBONE_LR_PHASE2},
        {"params": head,     "lr": HEAD_LR_PHASE2},
    ], weight_decay=WEIGHT_DECAY)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
    return opt, sch

scaler = torch.amp.GradScaler('cuda')
print("Model si configuratie gata. Faza 1 (backbone inghetat) incepe la antrenare.")


Antrenam pe: cuda
Clase: sanatos=48332, bolnav=48332
pos_weight dezactivat (set echilibrat).
Model si configuratie gata. Faza 1 (backbone inghetat) incepe la antrenare.


## (Opțional) Sanity-check: poate modelul să suprainvețe 100 de imagini?

Rulează celula asta o singură dată ca diagnostic. Dacă în ~20 de epoci pe 100 de imagini **nu** ajunge aproape de 100% acuratețe pe train, problema e în date / preprocesare / etichete, nu în model — și nu are rost să antrenezi ore întregi. Sari peste ea pentru antrenarea reală.

In [3]:
RUN_SANITY = True  # pune True ca sa rulezi diagnosticul

if RUN_SANITY:
    import copy
    small = Subset(train_ds, list(range(100)))
    small_loader = DataLoader(small, batch_size=16, shuffle=True, num_workers=0)
    m = copy.deepcopy(model)
    set_backbone_trainable(m, True)
    opt = optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0)
    for ep in range(20):
        m.train(); preds, tgts = [], []
        for images, labels in small_loader:
            images = images.to(device); labels = labels.float().to(device)
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                out = m(images).squeeze(1)
                loss = criterion(out, labels)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            preds += ((torch.sigmoid(out) > 0.5).int().detach().cpu().numpy().tolist())
            tgts  += labels.int().cpu().numpy().tolist()
        acc = accuracy_score(tgts, preds)
        print(f"sanity epoca {ep+1}: train_acc={acc:.3f}")
    print("Daca acc ajunge ~1.0 -> pipeline OK. Daca ramane ~0.5 -> problema in date/etichete.")
else:
    print("Sanity-check dezactivat (RUN_SANITY=False).")


sanity epoca 1: train_acc=0.680
sanity epoca 2: train_acc=0.980
sanity epoca 3: train_acc=1.000
sanity epoca 4: train_acc=1.000
sanity epoca 5: train_acc=1.000
sanity epoca 6: train_acc=1.000
sanity epoca 7: train_acc=1.000
sanity epoca 8: train_acc=1.000
sanity epoca 9: train_acc=1.000
sanity epoca 10: train_acc=1.000
sanity epoca 11: train_acc=1.000
sanity epoca 12: train_acc=1.000
sanity epoca 13: train_acc=1.000
sanity epoca 14: train_acc=1.000
sanity epoca 15: train_acc=1.000
sanity epoca 16: train_acc=1.000
sanity epoca 17: train_acc=1.000
sanity epoca 18: train_acc=1.000
sanity epoca 19: train_acc=1.000
sanity epoca 20: train_acc=1.000
Daca acc ajunge ~1.0 -> pipeline OK. Daca ramane ~0.5 -> problema in date/etichete.


## Antrenare (două faze + early stopping pe AUC)

In [ ]:
EPOCHS = 30
best_val_auc = -1.0
bad_epochs = 0

# Pornim in faza 1: backbone inghetat, doar capul
optimizer, scheduler = build_optimizer_phase1(model)
print("=== INCEPERE ANTRENARE - FAZA 1 (backbone inghetat) ===")

history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_auc": []}

for epoch in range(EPOCHS):
    # Tranzitia la faza 2
    if epoch == FREEZE_BACKBONE_EPOCHS:
        optimizer, scheduler = build_optimizer_phase2(model)
        print("=== FAZA 2: backbone dezghetat, fine-tuning complet cu LR diferentiat ===")

    # ---- TRAIN ----
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1}/{EPOCHS} [TRAIN]"):
        images = images.to(device)
        labels = labels.float().to(device)        # defensiv: BCEWithLogits cere float
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * images.size(0)

    # ---- VAL ----
    model.eval()
    val_loss = 0.0
    val_probs, val_targets = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1}/{EPOCHS} [VAL]"):
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_probs.extend(torch.sigmoid(outputs).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    avg_train_loss = train_loss / len(train_ds)
    avg_val_loss   = val_loss / len(val_ds)
    val_preds = (np.array(val_probs) > 0.5).astype(int)
    val_acc = accuracy_score(val_targets, val_preds)
    try:
        val_auc = roc_auc_score(val_targets, val_probs)
    except ValueError:
        val_auc = float("nan")

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)

    scheduler.step(val_auc)  # scheduler pe AUC (mode='max')

    print(f"Epoca {epoch+1}: T_Loss={avg_train_loss:.4f} | V_Loss={avg_val_loss:.4f} "
          f"| V_Acc={val_acc:.4f} | V_AUC={val_auc:.4f}")

    # Checkpoint + early stopping pe AUC
    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        bad_epochs = 0
        torch.save(model.state_dict(), 'efficientnet_b3_binar_best.pth')
        print(f"Model nou salvat! (Val AUC: {best_val_auc:.4f})")
    else:
        bad_epochs += 1
        print(f"Fara imbunatatire pe AUC: {bad_epochs}/{EARLY_STOPPING_PATIENCE}")
        if bad_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping.")
            break

print(f"Cel mai bun Val AUC: {best_val_auc:.4f}")


=== INCEPERE ANTRENARE - FAZA 1 (backbone inghetat) ===


Epoca 1/30 [VAL]: 100%|██████████| 416/416 [01:08<00:00,  6.11it/s]


Epoca 1: T_Loss=0.5724 | V_Loss=0.5240 | V_Acc=0.7762 | V_AUC=0.6508
Model nou salvat! (Val AUC: 0.6508)


Epoca 2/30 [VAL]: 100%|██████████| 416/416 [01:08<00:00,  6.07it/s]


Epoca 2: T_Loss=0.5664 | V_Loss=0.5203 | V_Acc=0.7737 | V_AUC=0.6530
Model nou salvat! (Val AUC: 0.6530)


...

Epoca 10/30 [VAL]: 100%|██████████| 416/416 [01:09<00:00,  5.97it/s]


Epoca 10: T_Loss=0.2086 | V_Loss=0.3264 | V_Acc=0.8795 | V_AUC=0.8621
Model nou salvat! (Val AUC: 0.8621)


Epoca 11/30 [VAL]: 100%|██████████| 416/416 [01:18<00:00,  5.32it/s]


Epoca 11: T_Loss=0.2004 | V_Loss=0.3280 | V_Acc=0.8780 | V_AUC=0.8659
Model nou salvat! (Val AUC: 0.8659)


Epoca 12/30 [TRAIN]:  31%|███       | 933/3021 [03:59<08:26,  4.12it/s] 

## Evaluare pe test (prag ales pe validare)

Alegem pragul de decizie pe setul de **validare** (Youden's J = max(TPR − FPR)) și îl aplicăm pe test. Raportăm și varianta la pragul standard 0.5 pentru comparație.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("EVALUARE PE SETUL DE TEST...")
model.load_state_dict(torch.load('efficientnet_b3_binar_best.pth'))
model.eval()

# 1. Recalculam probabilitatile pe VALIDARE ca sa alegem pragul
val_probs, val_targets = [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Val (prag)"):
        images = images.to(device)
        outputs = model(images).squeeze(1)
        val_probs.extend(torch.sigmoid(outputs).cpu().numpy())
        val_targets.extend(labels.numpy())

fpr, tpr, thr = roc_curve(val_targets, val_probs)
best_thr = float(thr[np.argmax(tpr - fpr)])
print(f"Prag optim ales pe validare (Youden's J): {best_thr:.3f}")

# 2. Probabilitati pe TEST
test_probs, test_targets = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testare"):
        images = images.to(device)
        outputs = model(images).squeeze(1)
        test_probs.extend(torch.sigmoid(outputs).cpu().numpy())
        test_targets.extend(labels.numpy())

test_probs = np.array(test_probs)
test_auc = roc_auc_score(test_targets, test_probs)
print(f"\nTest AUC: {test_auc:.4f}")

for label, t in [("prag 0.5", 0.5), (f"prag {best_thr:.3f} (din val)", best_thr)]:
    preds = (test_probs > t).astype(int)
    print(f"\nRAPORT - {label}:")
    print(classification_report(test_targets, preds, target_names=['Sanatos (0)', 'Bolnav (1)']))

# 3. Matricea de confuzie la pragul ales pe validare
preds = (test_probs > best_thr).astype(int)
cm = confusion_matrix(test_targets, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sanatos (0)', 'Bolnav (1)'],
            yticklabels=['Sanatos (0)', 'Bolnav (1)'],
            annot_kws={"size": 16})
plt.title('Matrice de Confuzie - EfficientNet-B3 (Binar)', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Eticheta Adevarata', fontsize=14, fontweight='bold')
plt.xlabel('Eticheta Prezisa', fontsize=14, fontweight='bold')
plt.xticks(fontsize=12); plt.yticks(fontsize=12, rotation=0)
plt.savefig('matrice_confuzie_binar.png', dpi=300, bbox_inches='tight')
plt.show()
